In [1]:
#if 'google.colab' in str(get_ipython()):
#    ! git clone -b main https://github.com/edsonportosilva/OptiCommPy
#    from os import chdir as cd
#    cd('/content/OptiCommPy/')
#    ! pip install .

#!pip install hdf5storage
#!pip install pykan

In [1]:
import numpy as np
import matplotlib.pyplot as plt

#import hdf5storage

from datetime               import datetime
from scipy.signal           import hilbert, firwin, welch
from scipy.constants        import pi
from scipy.io               import loadmat

from optic.comm.ofdm        import demodulateOFDM
from optic.comm.metrics     import fastBERcalc, calcEVM
from optic.utils            import parameters, dBm2W
from optic.dsp.core         import pnorm, signal_power, upsample, decimate, finddelay, firFilter, clockSamplingInterp
from optic.plot             import pconst

from proc_ofdm              import save_OFDM, tx_OFDM, rx_OFDM, prepare_data_training, plot_const, plot_sig, calculate_metrics, test_DPD_as_equalizer, calcACLR, sync_tx_frames
from dpd_mp                 import MP_filter, MP_training
from dpd_nn                 import NN_training, KAN_training

In [2]:
font = {'size':14}
plt.rc('font', **font)
plt.rcParams["font.family"] = "serif"

### 1 - Parameters initialization

In [3]:
paramOFDM = parameters()

# Modulation parameters
paramOFDM.mode     = "ARoF"
paramOFDM.modOrder = 64                # Modulation order
paramOFDM.modType  = 'qam'             # Modulation type
paramOFDM.numOFDMframes = 10           # Number of ofdm symbols generated

paramOFDM.bw  = 100e6                  # Channel bandwidth
paramOFDM.scs = 30e3                   # Subcarrier spacing

if paramOFDM.bw == 100e6 and paramOFDM.scs == 30e3:
    paramOFDM.Nfft = 2**12

elif paramOFDM.bw == 50e6 and paramOFDM.scs == 30e3:
    paramOFDM.Nfft  = 2**12

elif paramOFDM.bw == 10e6 and paramOFDM.scs == 30e3:
    paramOFDM.Nfft = 2**9

elif paramOFDM.bw == 400e6 and paramOFDM.scs == 480e3:
    paramOFDM.Nfft = 2**11

elif paramOFDM.bw == 100e6 and paramOFDM.scs == 480e3:
    paramOFDM.Nfft = 2**12

elif paramOFDM.bw == 400e6 and paramOFDM.scs == 1e6:
    paramOFDM.Nfft = 2**11

else:
    print("Check parameters for error in Nfft")

In [4]:
paramOFDM.G    = 32                    # Cyclic prefix length
paramOFDM.hermitSymmetry = False       # If true, the OFDM signal respects the Hermitian symmetry
paramOFDM.returnChannel  = False       # If true, the channel estimated response is returned

paramOFDM.saveTx       = False          # If true, the transmitted signal information is saved in a .mat file
paramOFDM.awg_model    = "Tektronix"   # "Keysight" "Tektronix" "R&S"
paramOFDM.awg_filepath = r"C:\Users\PC\Documents\Mestrado\DPD"
paramOFDM.tx_info_path = r"C:\Users\PC\Documents\Mestrado\DPD"
paramOFDM.seed = 3

if paramOFDM.hermitSymmetry:
    Ns = paramOFDM.Nfft//2 - 1
    paramOFDM.pilotCarriers = np.linspace(0, Ns - 1, 8, dtype = np.int64)
    paramOFDM.nullCarriers  = np.array([], dtype = np.int64)
else:
    Ns = paramOFDM.Nfft
    paramOFDM.pilotCarriers = np.linspace(0, Ns - 1, 8, dtype = np.int64)
    paramOFDM.nullCarriers  = np.array([paramOFDM.Nfft//2], dtype = np.int64)

# Pilot, null and information subcarriers
Np = paramOFDM.pilotCarriers.size
Nz = paramOFDM.nullCarriers.size
paramOFDM.Ni = Ns - Np - Nz

# RF carrier frequency
paramOFDM.fc = 3.55e9

# Devices sampling frequencies
paramOFDM.Fawg = 25e9
paramOFDM.Fdso = 20e9

paramOFDM.DPD_active = False

symbTx, sigTx, sigTx_RF = tx_OFDM(paramOFDM)

### 2 - Getting the experimental data for DPD training

In [5]:
filename = r"C:\Users\PC\Documents\Mestrado\DPD\Cooperacao_KTH\Dados_experimentais\att_0dB\dso_data1.mat"  # Path to the dso_data file
sigRx_RF = loadmat(filename)["dso_data"].ravel()

In [6]:
symbRx, sigRx, frames_rx = rx_OFDM(sigRx_RF, sigTx, paramOFDM, lpf = False, plot = False)

In [7]:
frames_rx

array([0, 1, 2, 3, 4, 5, 6])

### 3 - MP-DPD training

In [9]:
DPD_model = "MP"
SpS_DPD = 4
SpS_in  = paramOFDM.SpS

sigTx_sync = sync_tx_frames(sigTx, frames_rx, paramOFDM.SpS*(paramOFDM.Nfft + paramOFDM.G), paramOFDM.numOFDMframes)

sigRef, sigIn = prepare_data_training(sigTx_sync, sigRx, SpS_in, SpS_DPD, DPD_model, paramOFDM)

In [66]:
paramMP = parameters()
paramMP.model   = DPD_model
paramMP.SpS_DPD = SpS_DPD
paramMP.M = 4
paramMP.P = 4

paramMP.N = 20_000
paramMP.numIter = 5

paramMP.mu  = 1e-3
paramMP.lbd = 0.9999
paramMP.S   = np.eye(paramMP.P*paramMP.M, dtype = complex)*5e-2

paramMP.a_kl = None
paramMP.alg = "RLS"
paramMP.directLearn = False

paramMP.pgrsBar    = True
paramMP.showMSE    = False
paramMP.storeCoeff = True

DPD_MP, _, _, _ = MP_training(sigRef, paramMP, sigIn)

paramMP.DPD = DPD_MP

  0%|          | 0/5 [00:00<?, ?it/s]

In [67]:
symbTx_sync = sync_tx_frames(symbTx, frames_rx, paramOFDM.Ni, paramOFDM.numOFDMframes)

symbRx_MP, sigRx_MP = test_DPD_as_equalizer(sigRx, paramMP, paramOFDM, lpf = True, bw = 75e6)
EVM, BER, SNR = calculate_metrics(symbTx_sync, symbRx_MP, 500, paramOFDM)

print("As a equalizer, the DPD block based in MP has the following metric's performance:")
print(f"EVM = {EVM[0]:.4f} %")
print(f"SNR = {SNR[0]:.4f} dB")
print(f"BER = {BER[0]}")

As a equalizer, the DPD block based in MP has the following metric's performance:
EVM = 4.6493 %
SNR = 26.6593 dB
BER = 1.1858598076535391e-05


In [68]:
freq, P_sigRx    = welch(pnorm(sigRx)[0::SpS_in//SpS_DPD], fs = paramOFDM.Fawg / (SpS_in//SpS_DPD), nfft = 16*1024, return_onesided = False)
freq, P_sigRx_MP = welch(pnorm(sigRx_MP), fs = paramOFDM.Fawg / (SpS_in//SpS_DPD), nfft = 16*1024, return_onesided = False)

print(f"ACLR (w/o DPD) = {calcACLR(P_sigRx, freq, 65e6):.3f} dB")
print(f"ACLR (with MP-DPD) = {calcACLR(P_sigRx_MP, freq, 65e6):.3f} dB")

ACLR (w/o DPD) = -25.627 dB
ACLR (with MP-DPD) = -36.156 dB


In [ ]:
paramOFDM.saveTx     = True
paramOFDM.DPD_active = True
symbTx, sigTx_MP_DPD, sigTx_RF_MP_DPD = tx_OFDM(paramOFDM, paramMP)

### 4 - NN-DPD training

In [69]:
DPD_model = "NN"
SpS_DPD = 4
SpS_in  = paramOFDM.SpS
device = "cuda"

sigTx_sync = sync_tx_frames(sigTx, frames_rx, paramOFDM.SpS*(paramOFDM.Nfft + paramOFDM.G), paramOFDM.numOFDMframes)

sigRef, sigIn = prepare_data_training(sigTx_sync, sigRx, SpS_in, SpS_DPD, DPD_model, paramOFDM, device)

In [70]:
paramNN = parameters()

N = 100_000

paramNN.model   = DPD_model
paramNN.SpS_DPD = SpS_DPD

paramNN.divByL = 1
paramNN.trainTestFrac = 0.75
paramNN.batch_size = 1_000
paramNN.shuffle = False
paramNN.includeMemory = True
paramNN.Ntaps = 8
paramNN.K = 3
paramNN.augment = True

paramNN.layers = [(paramNN.K + 2)*paramNN.Ntaps, 16, 2]

paramNN.device = device
paramNN.lr = 5e-3
paramNN.epochs = 500
paramNN.activation = "relu"
paramNN.pgrsBar = True
paramNN.directLearn = False
paramNN.envelope = False

DPD_NN, train, test = NN_training(sigIn[0:N], sigRef[0:N], paramNN)

paramNN.DPD = DPD_NN

  0%|          | 0/500 [00:00<?, ?it/s]

In [71]:
symbTx_sync = sync_tx_frames(symbTx, frames_rx, paramOFDM.Ni, paramOFDM.numOFDMframes)

symbRx_NN, sigRx_NN = test_DPD_as_equalizer(sigRx, paramNN, paramOFDM, lpf = True, bw = 75e6)
EVM, BER, SNR = calculate_metrics(symbTx_sync, symbRx_NN, 500, paramOFDM)

print("As a equalizer, the DPD block based in NN has the following metric's performance:")
print(f"EVM = {EVM[0]:.4f} %")
print(f"SNR = {SNR[0]:.4f} dB")
print(f"BER = {BER[0]}")

As a equalizer, the DPD block based in NN has the following metric's performance:
EVM = 5.6434 %
SNR = 24.9767 dB
BER = 0.00034389934421952636


In [72]:
freq, P_sigRx    = welch(pnorm(sigRx)[0::SpS_in//SpS_DPD], fs = paramOFDM.Fawg / (SpS_in//SpS_DPD), nfft = 16*1024, return_onesided = False)
freq, P_sigRx_NN = welch(pnorm(sigRx_NN), fs = paramOFDM.Fawg / (SpS_in//SpS_DPD), nfft = 16*1024, return_onesided = False)

print(f"ACLR (w/o DPD) = {calcACLR(P_sigRx, freq, 65e6):.3f} dB")
print(f"ACLR (with NN-DPD) = {calcACLR(P_sigRx_NN, freq, 65e6):.3f} dB")

ACLR (w/o DPD) = -25.627 dB
ACLR (with NN-DPD) = -35.831 dB


In [16]:
paramOFDM.saveTx     = True
paramOFDM.DPD_active = True
symbTx, sigTx_NN_DPD, sigTx_RF_NN_DPD = tx_OFDM(paramOFDM, paramNN)

### 5 - KAN-DPD training

In [73]:
DPD_model = "KAN"
SpS_DPD = 4
SpS_in  = paramOFDM.SpS
device = "cuda"

sigTx_sync = sync_tx_frames(sigTx, frames_rx, paramOFDM.SpS*(paramOFDM.Nfft + paramOFDM.G), paramOFDM.numOFDMframes)

sigRef, sigIn = prepare_data_training(sigTx_sync, sigRx, SpS_in, SpS_DPD, DPD_model, paramOFDM, device)

In [74]:
paramKAN = parameters()

N = 50_000

paramKAN.model   = DPD_model
paramKAN.SpS_DPD = SpS_DPD

paramKAN.divByL = 1
paramKAN.trainTestFrac = 0.75
paramKAN.batch_size = 1_000
paramKAN.shuffle = False
paramKAN.includeMemory = True
paramKAN.Ntaps = 2
paramKAN.K = 0
paramKAN.augment = False

paramKAN.layers = [2*paramKAN.Ntaps, 5, 2]

paramKAN.device = device
paramKAN.lr = 5e-3
paramKAN.epochs = 200
paramKAN.pgrsBar = True
paramKAN.directLearn = False

paramKAN.k = 3
paramKAN.grid = 5
paramKAN.seed = 0
paramKAN.pruning_epochs = []
paramKAN.envelope = False

DPD_KAN, _, _ = KAN_training(sigIn[0:N], sigRef[0:N], paramKAN)

paramKAN.DPD = DPD_KAN

  0%|          | 0/200 [00:00<?, ?it/s]

In [75]:
symbTx_sync = sync_tx_frames(symbTx, frames_rx, paramOFDM.Ni, paramOFDM.numOFDMframes)

symbRx_KAN, sigRx_KAN = test_DPD_as_equalizer(sigRx, paramKAN, paramOFDM, lpf = True, bw = 75e6)
EVM, BER, SNR = calculate_metrics(symbTx_sync, symbRx_KAN, 500, paramOFDM)

print("As a equalizer, the DPD block based in KAN has the following metric's performance:")
print(f"EVM = {EVM[0]:.4f} %")
print(f"SNR = {SNR[0]:.4f} dB")
print(f"BER = {BER[0]}")

As a equalizer, the DPD block based in KAN has the following metric's performance:
EVM = 5.6409 %
SNR = 24.9862 dB
BER = 0.00037354583941086485


In [76]:
freq, P_sigRx    = welch(pnorm(sigRx)[0::SpS_in//SpS_DPD], fs = paramOFDM.Fawg / (SpS_in//SpS_DPD), nfft = 16*1024, return_onesided = False)
freq, P_sigRx_KAN = welch(pnorm(sigRx_KAN), fs = paramOFDM.Fawg / (SpS_in//SpS_DPD), nfft = 16*1024, return_onesided = False)

print(f"ACLR (w/o DPD) = {calcACLR(P_sigRx, freq, 65e6):.3f} dB")
print(f"ACLR (with KAN-DPD) = {calcACLR(P_sigRx_KAN, freq, 65e6):.3f} dB")

ACLR (w/o DPD) = -25.627 dB
ACLR (with KAN-DPD) = -34.949 dB


In [ ]:
paramOFDM.saveTx     = True
paramOFDM.DPD_active = True
symbTx, sigTx_KAN_DPD, sigTx_RF_KAN_DPD = tx_OFDM(paramOFDM, paramKAN)